# GPS OD-Flow Training (Our Model)

Train **our GPS-based model** (GPS-GNN encoder + TransFlower/Bilinear decoder)
with different configurations. Weights and metrics saved to `results/`
and consumed by `benchmark.ipynb` for comparison against baselines.

**Architecture:** GPS-GNN encoder (GINEConv + multi-head attention) with two decoder options:
- `transflower`: TransFlower decoder (Transformer-based)
- `bilinear`: Bilinear decoder
- `gravity_guided`: Gravity-guided decoder

**Ablation axes:** `pe_type`, `loss_type`, `gps_norm_type`, `use_log_transform`, `use_rle`, `use_dest_sampling`

The GPS-GAN section adds `training_mode='gan'`, an OD random-walk discriminator, and WGAN-GP adversarial training.


In [1]:
import sys, os, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path('.').resolve()))
warnings.filterwarnings('ignore')

from models.GPS.config import (
    TrainingConfig, device, cleanup_gpu, MC_EPOCHS,
    SINGLE_CITY_ID, SINGLE_CITY_IDS, MULTI_CITY_IDS, RESULTS_DIR,
    METRICS_CSV, METRICS_VAL_LOSS_CSV, METRICS_CPC_NZ_BEST_CSV, WEIGHTS_DIR,
    ensure_dirs,
)
from models.GPS.data_load import prepare_single_city_data, prepare_multi_city_data
from models.GPS.main import train_single_city, train_multi_city
from models.GPS.metrics import evaluate_full_matrix, compute_metrics
from benchmarking.repeats import single_city_run_id

ensure_dirs()
print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")


Device: cuda:1
Results dir: /home/rk/Документы/Python Projects/GSP_OD_Prediction/results


## GPS Configurations

Edit this cell to select which experiments to run.

In [ ]:
from dataclasses import replace

# Config selectors -----------------------------------------------------------
# Empty RUN_ONLY_CONFIG_IDS means "use all enabled configs".
# To run only the regularized multi-city candidates, set:
# RUN_ONLY_CONFIG_IDS = {'MC_GG_HUBER_LOG_NORM_GN_RWPE', 'MC_GG_CE_GN_RWPE_WD'}
ENABLE_SC_CONFIGS = True
ENABLE_MC_MIRRORED_CONFIGS = True
ENABLE_MC_STRONG_CONFIGS = True
RUN_ONLY_CONFIG_IDS = set()
DISABLED_CONFIG_IDS = set()

# Only combinations that are implemented consistently in GPS loss + inference.
# - CE / Focal are normalized distribution losses; log is intentionally off.
# - Huber / Multitask support raw, raw+log, and normalized+log_norm.
# - ZINB has its own count-style decoder path; keep it raw and no-log.
BASE_ZERO_KW = dict(include_zero_pairs=True, zero_pair_ratio=0.3)
DECODER_KW = {
    'BL': dict(decoder_type='bilinear', use_dest_sampling=False),
    'TF': dict(decoder_type='transflower', use_dest_sampling=True),
    'GG': dict(decoder_type='gravity_guided', use_dest_sampling=False),
}
CORE_DECODERS = ('BL', 'TF')
LOGGABLE_LOSSES = {'huber', 'multitask'}
DISTRIBUTION_LOSSES = {'ce', 'ce_old', 'focal'}
IMPLEMENTED_LOSSES = {'ce', 'ce_old', 'focal', 'huber', 'multitask', 'zinb'}


LOSS_ABLATIONS = [
    # Baseline / loss ablations
    ('CE', 'CE', dict(loss_type='ce', prediction_mode='normalized')),
    ('FOCAL', 'Focal', dict(loss_type='focal', prediction_mode='normalized', focal_gamma=2.0)),
    ('HUBER_RAW', 'HuberRaw', dict(loss_type='huber', prediction_mode='raw')),
    ('HUBER_LOG', 'Huber', dict(loss_type='huber', prediction_mode='raw', use_log_transform=True)),
    ('HUBER_LOG_NORM', 'Huber', dict(loss_type='huber', prediction_mode='normalized', use_log_transform=True)),
    ('MULTITASK_RAW', 'MultitaskRaw', dict(loss_type='multitask', prediction_mode='raw')),
    ('MULTITASK_LOG_NORM', 'Multitask', dict(loss_type='multitask', prediction_mode='normalized', use_log_transform=True)),
    ('ZINB_RAW', 'ZINBRaw', dict(loss_type='zinb', prediction_mode='raw')),
]

MODULE_ABLATIONS = [
    # One-module ablations on top of the CE baseline.
    ('RWPE', 'RWPE', dict(loss_type='ce', prediction_mode='normalized', pe_type='rwpe')),
    ('LAPE', 'LaPE', dict(loss_type='ce', prediction_mode='normalized', pe_type='lape')),
    ('GRAPH_NORM', 'GraphNorm', dict(loss_type='ce', prediction_mode='normalized', gps_norm_type='graph_norm')),
    ('RLE', 'RLE', dict(loss_type='ce', prediction_mode='normalized', use_rle=True)),
]

ABLATIONS = LOSS_ABLATIONS + MODULE_ABLATIONS


def _validate_supported_combo(cfg):
    """Fail fast for configs that TrainingConfig accepts but GPS code does not support."""
    if cfg.loss_type not in IMPLEMENTED_LOSSES:
        raise ValueError(f"{cfg.loss_type=} is not implemented in models/GPS/loss.py")

    if cfg.loss_type in DISTRIBUTION_LOSSES:
        if cfg.prediction_mode != 'normalized':
            raise ValueError(f"{cfg.loss_type} must use prediction_mode='normalized'")
        if cfg.use_log_transform:
            raise ValueError(f"{cfg.loss_type} ignores/breaks log transform; keep use_log_transform=False")

    if cfg.loss_type == 'zinb':
        if cfg.prediction_mode != 'raw':
            raise ValueError("zinb uses its own count decoder path; keep prediction_mode='raw'")
        if cfg.use_log_transform:
            raise ValueError("zinb already predicts counts; keep use_log_transform=False")

    if cfg.use_log_transform and cfg.loss_type not in LOGGABLE_LOSSES:
        raise ValueError(f"log transform is supported only for {sorted(LOGGABLE_LOSSES)}")

    return cfg


def _gps_cfg(decoder_code, allow_sample_override=False, **kw):
    """Build a GPS config and enforce the ablation invariants."""
    defaults = dict(
        encoder_type='gps',
        pe_type=None,
        gps_norm_type='batch_norm',
        loss_type='ce',
        prediction_mode='normalized',
        use_log_transform=False,
        use_rle=False,
        **BASE_ZERO_KW,
        **DECODER_KW[decoder_code],
    )
    defaults.update(kw)

    # Keep zeros fixed everywhere; keep sampling policy fixed by decoder.
    defaults.update(BASE_ZERO_KW)
    if decoder_code in ('BL', 'GG'):
        defaults['use_dest_sampling'] = False
    elif decoder_code == 'TF' and not allow_sample_override:
        defaults['use_dest_sampling'] = True

    return _validate_supported_combo(TrainingConfig(**defaults))


def _run_id(scope, decoder_code, ablation_code):
    return f'{scope}_{decoder_code}_{ablation_code}'


def _run_name(scope, decoder_code, ablation_name, cfg):
    sample = 'sample_on' if cfg.use_dest_sampling else 'sample_off'
    pe_name = 'none' if cfg.pe_type is None else cfg.pe_type
    log_tag = '+log_norm' if cfg.use_log_transform and cfg.prediction_mode == 'normalized' else ('+log' if cfg.use_log_transform else '')
    return f'{scope} GPS+{decoder_code}+{ablation_name}{log_tag} | pe={pe_name} | zeros=0.3 | {sample}'


def _build_sc_configs():
    configs = {}
    for decoder_code in CORE_DECODERS:
        for ablation_code, ablation_name, overrides in ABLATIONS:
            cfg = _gps_cfg(decoder_code, **overrides)
            rid = _run_id('SC', decoder_code, ablation_code)
            configs[rid] = (_run_name('SC', decoder_code, ablation_name, cfg), cfg)

    gg_cfg = _gps_cfg('GG', loss_type='ce', prediction_mode='normalized')
    configs[_run_id('SC', 'GG', 'CE')] = (
        _run_name('SC', 'GG', 'CE', gg_cfg),
        gg_cfg,
    )

    cfg = _gps_cfg(
        'TF',
        allow_sample_override=True,
        loss_type='ce',
        prediction_mode='normalized',
        use_dest_sampling=False,
    )
    configs[_run_id('SC', 'TF', 'SAMPLE_FALSE')] = (
        _run_name('SC', 'TF', 'sample_false', cfg),
        cfg,
    )
    return configs


def _build_mc_configs_from_sc(sc_configs):
    return {
        rid.replace('SC_', 'MC_', 1): (name.replace('SC ', 'MC ', 1), replace(cfg, mc_epochs=MC_EPOCHS))
        for rid, (name, cfg) in sc_configs.items()
    }


def _mc_strong_cfg(decoder_code, **kw):
    """Regularized MC-only configs aimed at city-to-city generalization."""
    defaults = dict(
        pe_type='rwpe',
        gps_norm_type='graph_norm',
        learning_rate=1.5e-4,
        weight_decay=3e-4,
        patience=25,
        mc_epochs=200,
    )
    defaults.update(kw)
    return _gps_cfg(decoder_code, **defaults)


def _build_mc_strong_configs():
    return {
        'MC_GG_HUBER_LOG_NORM_GN_RWPE': (
            'MC GPS+GG+Huber+log_norm+GraphNorm+RWPE | reg=wd3e-4 lr1.5e-4 | zeros=0.3 | sample_off',
            _mc_strong_cfg(
                'GG',
                loss_type='huber',
                prediction_mode='normalized',
                use_log_transform=True,
            ),
        ),
        'MC_GG_CE_GN_RWPE_WD': (
            'MC GPS+GG+CE+GraphNorm+RWPE | reg=wd3e-4 lr1.5e-4 | zeros=0.3 | sample_off',
            _mc_strong_cfg(
                'GG',
                loss_type='ce',
                prediction_mode='normalized',
            ),
        ),
        'MC_GG_FOCAL_GN_RWPE_WD': (
            'MC GPS+GG+Focal+GraphNorm+RWPE | reg=wd3e-4 lr1.5e-4 | zeros=0.3 | sample_off',
            _mc_strong_cfg(
                'GG',
                loss_type='focal',
                prediction_mode='normalized',
                focal_gamma=1.0,
            ),
        ),
        'MC_BL_HUBER_LOG_NORM_GN_RWPE_WD': (
            'MC GPS+BL+Huber+log_norm+GraphNorm+RWPE | reg=wd3e-4 lr1.5e-4 | zeros=0.3 | sample_off',
            _mc_strong_cfg(
                'BL',
                loss_type='huber',
                prediction_mode='normalized',
                use_log_transform=True,
            ),
        ),
        'MC_TF_CE_GN_RWPE_WD': (
            'MC GPS+TF+CE+GraphNorm+RWPE | reg=wd3e-4 lr1.5e-4 | zeros=0.3 | sample_on',
            _mc_strong_cfg(
                'TF',
                loss_type='ce',
                prediction_mode='normalized',
            ),
        ),
    }


def _apply_config_filters(configs):
    if DISABLED_CONFIG_IDS:
        configs = {rid: item for rid, item in configs.items() if rid not in DISABLED_CONFIG_IDS}
    if RUN_ONLY_CONFIG_IDS:
        configs = {rid: item for rid, item in configs.items() if rid in RUN_ONLY_CONFIG_IDS}
    return configs


_all_sc_configs = _build_sc_configs()
SC_CONFIGS = _apply_config_filters(_all_sc_configs if ENABLE_SC_CONFIGS else {})
MC_CONFIGS = _build_mc_configs_from_sc(_all_sc_configs) if ENABLE_MC_MIRRORED_CONFIGS else {}
if ENABLE_MC_STRONG_CONFIGS:
    MC_CONFIGS.update(_build_mc_strong_configs())
MC_CONFIGS = _apply_config_filters(MC_CONFIGS)

print(f"Single-city configs: {len(SC_CONFIGS)}")
print(f"Multi-city configs:  {len(MC_CONFIGS)}")
print(f"TF configs: {sum(1 for k in SC_CONFIGS if '_TF_' in k)}")
print(f"BL configs: {sum(1 for k in SC_CONFIGS if '_BL_' in k)}")
print(f"GG configs: {sum(1 for k in SC_CONFIGS if '_GG_' in k)}")
if RUN_ONLY_CONFIG_IDS:
    print(f"RUN_ONLY_CONFIG_IDS active: {sorted(RUN_ONLY_CONFIG_IDS)}")
if DISABLED_CONFIG_IDS:
    print(f"DISABLED_CONFIG_IDS active: {sorted(DISABLED_CONFIG_IDS)}")
print("\nSingle-city:")
for rid, (run_name, cfg) in SC_CONFIGS.items():
    print(f"  {rid:<32} {run_name:<84} {cfg.describe()}")
print("\nMulti-city:")
for rid, (run_name, cfg) in MC_CONFIGS.items():
    print(f"  {rid:<32} {run_name:<84} {cfg.describe()}")


## Single-City Training

Three explicit cells below train the same single-city configs on three separate cities.
Each run is saved with a `__city_<city_id>` suffix, so weights and metrics do not overwrite each other.


In [ ]:
# Single-city data loading with city+pe caching
SC_CITY_1, SC_CITY_2, SC_CITY_3 = SINGLE_CITY_IDS
print(f"Single-city benchmark cities: {SINGLE_CITY_IDS}")

_sc_cache = {}

def get_sc_data(area_id, pe_type='rwpe'):
    key = (area_id, pe_type)
    if key not in _sc_cache:
        print(f"  Loading single-city data (city={area_id}, pe_type={pe_type})...")
        _sc_cache[key] = prepare_single_city_data(area_id=area_id, pe_type=pe_type)
        n_nodes = _sc_cache[key]['num_nodes']
        train_pairs = _sc_cache[key]['train_mask'].sum()
        print(f"  N={n_nodes}, train_pairs={train_pairs}")
    return _sc_cache[key]


def train_sc_city(area_id):
    city_results = {}
    print(f"\n{'=' * 70}\n  SINGLE-CITY TRAINING FOR {area_id}\n{'=' * 70}")
    for base_run_id, (run_name, cfg) in SC_CONFIGS.items():
        run_id = single_city_run_id(base_run_id, area_id)
        city_data = get_sc_data(area_id, cfg.pe_type)
        try:
            result = train_single_city(
                run_id,
                f"{run_name} [{area_id}]",
                cfg,
                city_data=city_data,
            )
            city_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()
    print(f"\nCompleted: {len(city_results)} single-city runs for {area_id}")
    return city_results


In [ ]:
sc_results_city_1 = train_sc_city(SC_CITY_1)

In [ ]:
sc_results_city_2 = train_sc_city(SC_CITY_2)

In [ ]:
sc_results_city_3 = train_sc_city(SC_CITY_3)

sc_results = {}
sc_results.update(sc_results_city_1)
sc_results.update(sc_results_city_2)
sc_results.update(sc_results_city_3)

print(f"\nCombined single-city runs: {len(sc_results)}")
print(sorted(sc_results.keys()))


## Multi-City Training

In [6]:
# ── Multi-city data loading with pe_type caching ─────────────────────────────
_mc_cache = {}

def get_mc_data(pe_type='rwpe'):
    if pe_type not in _mc_cache:
        print(f"  Loading multi-city data (pe_type={pe_type})...")
        city_data_dict, train_ids, val_ids, test_ids = prepare_multi_city_data(pe_type=pe_type)
        _mc_cache[pe_type] = (city_data_dict, train_ids, val_ids, test_ids)
        print(f"  Train: {train_ids}  Val: {val_ids}  Test: {test_ids}")
    return _mc_cache[pe_type]


In [7]:
mc_results = {}

for run_id, (run_name, cfg) in MC_CONFIGS.items():
    city_data_dict, train_ids, val_ids, test_ids = get_mc_data(cfg.pe_type)
    try:
        result = train_multi_city(
            run_id, run_name, cfg,
            city_data_dict=city_data_dict,
            train_city_ids=train_ids,
            val_city_ids=val_ids,
        )
        mc_results[run_id] = result
    except Exception as e:
        print(f"  ERROR {run_id}: {e}")
    finally:
        cleanup_gpu()

print(f"\nCompleted: {len(mc_results)} multi-city runs")


  Loading multi-city data (pe_type=lape)...
  17031: N=1318
  48201: N=786
  04013: N=916
  06073: N=627
  06059: N=582
  36047: N=761
  12086: N=518
  48113: N=529
  06065: N=453
  36081: N=669
  06065: N=453
  48201: N=786
  36047: N=761
  17031: N=1318
  48113: N=529
  04013: N=916
  36081: N=669
  06059: N=582
  06073: N=627
  12086: N=518
  Train: ['06065', '48201', '36047', '17031', '48113', '04013', '36081', '06059']  Val: ['06073']  Test: ['12086']

  MC BL CE+LaPE+GN+Log+RLE+ZP+noDS
  GPS+bilinear+ce+normalized | pe=lape | norm=graph_norm | log | RLE | zeros=True samp=False
  Params: 748,094
    1/200  train=6.5029  val=6.7457  CPC_full=0.3119  CPC_nz=0.4166  26.2s *
    5/200  train=5.9747  val=6.4009  CPC_full=0.3227  CPC_nz=0.4189  26.0s
   10/200  train=5.6773  val=6.4064  CPC_full=0.3270  CPC_nz=0.4134  26.2s
   15/200  train=5.5111  val=6.4102  CPC_full=0.3283  CPC_nz=0.4145  26.4s
   20/200  train=5.4276  val=6.8968  CPC_full=0.2849  CPC_nz=0.3573  26.1s
   25/200  trai

KeyboardInterrupt: 

## GPS-GAN Training

These runs keep the GPS generator and add an OD random-walk TCN discriminator. `SC_GG_GAN_CE_RWPE_GN` is the gravity-guided ODGN-style run; `SC_BL_GAN_CE_RWPE_GN` is a lighter GAT-GAN-like ablation without the gravity decoder.


In [ ]:
# GPS-GAN configs -----------------------------------------------------------
ENABLE_GPS_GAN_SC = True
ENABLE_GPS_GAN_MC = False  # optional; GAN multi-city is much heavier
GPS_GAN_CITY_IDS = [SC_CITY_1]
GPS_GAN_RUN_ONLY = {'SC_GG_GAN_CE_RWPE_GN'}  # empty set means run all SC GAN configs
GPS_GAN_MC_RUN_ONLY = set()

def _gps_gan_cfg(decoder_type='gravity_guided', **kw):
    defaults = dict(
        encoder_type='gps',
        decoder_type=decoder_type,
        training_mode='gan',
        loss_type='ce',
        prediction_mode='normalized',
        pe_type='rwpe',
        gps_norm_type='graph_norm',
        use_dest_sampling=False,
        include_zero_pairs=True,
        zero_pair_ratio=0.3,
        learning_rate=1.5e-4,
        discriminator_lr=1.5e-4,
        weight_decay=3e-4,
        epochs=60,
        patience=20,
        gan_pretrain_epochs=10,
        adv_weight=0.03,
        gan_n_critic=1,
        gan_walk_len=64,
        gan_walk_batch_size=64,
        gan_disc_layers=4,
        gan_disc_dropout=0.05,
    )
    defaults.update(kw)
    return TrainingConfig(**defaults)

GPS_GAN_SC_CONFIGS = {
    'SC_GG_GAN_CE_RWPE_GN': (
        'SC GPS+GG+CE+GAN | pe=rwpe | graph_norm | WGAN-GP',
        _gps_gan_cfg('gravity_guided'),
    ),
    'SC_BL_GAN_CE_RWPE_GN': (
        'SC GPS+BL+CE+GAN | pe=rwpe | graph_norm | GAT-GAN-like',
        _gps_gan_cfg('bilinear'),
    ),
}
if GPS_GAN_RUN_ONLY:
    GPS_GAN_SC_CONFIGS = {rid: item for rid, item in GPS_GAN_SC_CONFIGS.items() if rid in GPS_GAN_RUN_ONLY}

GPS_GAN_MC_CONFIGS = {
    'MC_GG_GAN_CE_RWPE_GN': (
        'MC GPS+GG+CE+GAN | pe=rwpe | graph_norm | WGAN-GP',
        replace(_gps_gan_cfg('gravity_guided'), mc_epochs=80),
    ),
}
if GPS_GAN_MC_RUN_ONLY:
    GPS_GAN_MC_CONFIGS = {rid: item for rid, item in GPS_GAN_MC_CONFIGS.items() if rid in GPS_GAN_MC_RUN_ONLY}

print(f"GPS-GAN single-city configs: {len(GPS_GAN_SC_CONFIGS)}")
for rid, (run_name, cfg) in GPS_GAN_SC_CONFIGS.items():
    print(f"  {rid:<28} {run_name:<68} {cfg.describe()}")
print(f"GPS-GAN multi-city configs:  {len(GPS_GAN_MC_CONFIGS)}")


In [ ]:
gps_gan_results = {}

def train_gps_gan_sc_city(area_id):
    city_results = {}
    print(f"\n{'=' * 70}\n  GPS-GAN TRAINING FOR {area_id}\n{'=' * 70}")
    for base_run_id, (run_name, cfg) in GPS_GAN_SC_CONFIGS.items():
        run_id = single_city_run_id(base_run_id, area_id)
        city_data = get_sc_data(area_id, cfg.pe_type)
        try:
            result = train_single_city(
                run_id,
                f"{run_name} [{area_id}]",
                cfg,
                city_data=city_data,
            )
            city_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()
    print(f"\nCompleted: {len(city_results)} GPS-GAN runs for {area_id}")
    return city_results

if ENABLE_GPS_GAN_SC:
    for area_id in GPS_GAN_CITY_IDS:
        gps_gan_results.update(train_gps_gan_sc_city(area_id))

if ENABLE_GPS_GAN_MC:
    for run_id, (run_name, cfg) in GPS_GAN_MC_CONFIGS.items():
        city_data_dict, train_ids, val_ids, test_ids = get_mc_data(cfg.pe_type)
        try:
            result = train_multi_city(
                run_id,
                run_name,
                cfg,
                city_data_dict=city_data_dict,
                train_city_ids=train_ids,
                val_city_ids=val_ids,
                test_city_ids=test_ids,
            )
            gps_gan_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()

print(f"\nCompleted GPS-GAN runs: {len(gps_gan_results)}")
print(sorted(gps_gan_results.keys()))


## GMEL-GPS Training

Single-city GMEL-GPS variants are also trained separately for the same three cities,
with per-city run IDs so `benchmark.ipynb` can load them city by city.


In [ ]:
from models.GPS.config import TrainingConfig
from models.GMEL_GPS.main import train as gmel_gps_train

# GMEL_GPS uses TrainingConfig with encoder_type='gmel_gps', decoder_type='gbrt'
def _gmel_gps(**kw):
    defaults = dict(encoder_type='gps', decoder_type='gbrt', loss_type='ce', prediction_mode='normalized')
    defaults.update(kw)
    return TrainingConfig(**defaults)

GMEL_GPS_CONFIGS = {
    'GMEL_GPS_rwpe':    ('GMEL-GPS RWPE  BN', _gmel_gps(pe_type='rwpe', gps_norm_type='batch_norm')),
    'GMEL_GPS_lape':    ('GMEL-GPS LaPE  BN', _gmel_gps(pe_type='lape', gps_norm_type='batch_norm')),
    'GMEL_GPS_nope':    ('GMEL-GPS No-PE BN', _gmel_gps(pe_type=None,   gps_norm_type='batch_norm')),
    'GMEL_GPS_rwpe_gn': ('GMEL-GPS RWPE  GN', _gmel_gps(pe_type='rwpe', gps_norm_type='graph_norm')),
}
print(f"GMEL_GPS configs: {len(GMEL_GPS_CONFIGS)}")
for rid, (name, cfg) in GMEL_GPS_CONFIGS.items():
    print(f"  {rid:<22}  {cfg.describe()}")


In [ ]:
# GMEL-GPS data loading with city+pe caching
_gg_cache = {}

def get_gg_data(area_id, pe_type):
    key = (area_id, pe_type)
    if key not in _gg_cache:
        print(f"  Loading GMEL-GPS data (city={area_id}, pe_type={pe_type})...")
        _gg_cache[key] = prepare_single_city_data(area_id=area_id, pe_type=pe_type)
        n_nodes = _gg_cache[key]['num_nodes']
        train_pairs = _gg_cache[key]['train_mask'].sum()
        print(f"  N={n_nodes}, train_pairs={train_pairs}")
    return _gg_cache[key]


def train_gmel_gps_city(area_id):
    city_results = {}
    print(f"\n{'=' * 70}\n  GMEL-GPS TRAINING FOR {area_id}\n{'=' * 70}")
    for base_run_id, (run_name, cfg) in GMEL_GPS_CONFIGS.items():
        run_id = single_city_run_id(base_run_id, area_id)
        city_data = get_gg_data(area_id, cfg.pe_type)
        try:
            result = gmel_gps_train(run_id, f"{run_name} [{area_id}]", cfg, city_data)
            city_results[run_id] = result
        except Exception as exc:
            print(f"  ERROR {run_id}: {exc}")
        finally:
            cleanup_gpu()
    print(f"\nCompleted: {len(city_results)} GMEL-GPS runs for {area_id}")
    return city_results


In [ ]:
gmel_gps_results_city_1 = train_gmel_gps_city(SC_CITY_1)

In [ ]:
gmel_gps_results_city_2 = train_gmel_gps_city(SC_CITY_2)

In [ ]:
gmel_gps_results_city_3 = train_gmel_gps_city(SC_CITY_3)

gmel_gps_results = {}
gmel_gps_results.update(gmel_gps_results_city_1)
gmel_gps_results.update(gmel_gps_results_city_2)
gmel_gps_results.update(gmel_gps_results_city_3)

print(f"\nCombined GMEL-GPS runs: {len(gmel_gps_results)}")
print(sorted(gmel_gps_results.keys()))


In [ ]:
# GMEL-GPS summary is shown below together with the other result tables and plots.


## Results Summary

In [ ]:
def print_results_table(results, label):
    if not results:
        print(f"  No {label} results.")
        return
    print(f"\n{'='*70}")
    print(f"  {label} Results")
    print(f"{'='*70}")
    print(f"  {'Run ID':<28} {'CPC_full':>9} {'CPC_nz':>9} {'CPC_test':>9} {'MAE':>9}  Status")
    print(f"  {'-'*76}")
    for rid, r in results.items():
        mf = r.get('metrics_full', {})
        mnz = r.get('metrics_nonzero', {})
        mt = r.get('metrics_test_pairs', {})
        status = r.get('status', '?')
        print(f"  {rid:<28} {mf.get('CPC', float('nan')):>9.4f} "
              f"{mnz.get('CPC', float('nan')):>9.4f} "
              f"{mt.get('CPC', float('nan')):>9.4f} "
              f"{mf.get('MAE', float('nan')):>9.4f}  {status}")

print_results_table(sc_results, "Single-City")
print_results_table(mc_results, "Multi-City")
print_results_table(globals().get('gps_gan_results', {}), "GPS-GAN")
print_results_table(gmel_gps_results, "GMEL-GPS")

# Show saved metrics CSVs
for metrics_path in (METRICS_VAL_LOSS_CSV, METRICS_CPC_NZ_BEST_CSV, METRICS_CSV):
    if metrics_path.exists():
        df = pd.read_csv(metrics_path)
        print(f"\n  Total rows in {metrics_path.name}: {len(df)}")
print(f"  Saved weights: {len(list(WEIGHTS_DIR.glob('*.pt')))}")


In [ ]:
# Training curves
def plot_histories(results, title):
    if not results:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    for rid, r in results.items():
        h = r.get('history', {})
        if not h:
            continue
        axes[0].plot(h.get('train_loss', []), label=rid, alpha=0.8)
        axes[1].plot(h.get('val_loss', []), label=rid, alpha=0.8)
        axes[2].plot(h.get('val_cpc_full', []), label=rid, alpha=0.8)
    axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
    axes[2].set_title('CPC (full)'); axes[2].set_xlabel('Epoch')
    for ax in axes:
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_histories(sc_results, "Single-City Training Histories")
plot_histories(mc_results, "Multi-City Training Histories")
plot_histories(globals().get('gps_gan_results', {}), "GPS-GAN Training Histories")
plot_histories(gmel_gps_results, "GMEL-GPS Training Histories")
